In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch import device
import numpy as np
import pandas as pd
import json
import os

In [2]:
class Raphyxr(nn.Module):
    def __init__(self, nb_entrees):
        super().__init__()

        self.reseau = nn.Sequential(
            nn.Linear(nb_entrees, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.reseau(x)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = Raphyxr(nb_entrees=8).to(device)


criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

if os.path.exists("checkpoint.pt"):
    checkpoint = torch.load("checkpoint.pt", map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    epoch_depart = checkpoint["epoch"] + 1
else:
    epoch_depart = 0

In [4]:
def selectionner(y):
    """ selectionne une ligne de données"""
    return df.iloc[y].tolist()


In [5]:
def encode(colonnes, chemin_json):
    with open(chemin_json, "r", encoding="utf-8") as f:
        dico = json.load(f)

    # Évite les doublons d'ID si le dictionnaire a été modifié.
    prochain_id = max(map(int, dico.values()), default=0) + 1
    ids = [] # liste des encodages 
    modifie = False # true: on modifie, false, il n'y a rien a modifier

    for token in colonnes: # pour chaque colonne de la ligne
        if not isinstance(token, (int, float, np.integer, np.floating)):
            token = str(token).strip() # on enleve les espaces avant et apres la valeur
            if token not in dico: # si le token n'est pas dans le dico
                dico[token] = prochain_id # on ajoute le mot a la liste des mots a ajouter 
                prochain_id += 1 #et on ajoute 1 pour l'id suivant
                modifie = True # on indique qu'on a des modifs a faire 
            ids.append(int(dico[token])) # on ajoute l'id a la liste des mots decodés

        else:
            ids.append(float(token)) # si c'est un nombre, on l'ajoute directement a la liste des mots decodés

    if modifie: # si il y a besoin de modifier
        with open(chemin_json, "w", encoding="utf-8") as f: # on ouvre le fichier json pour ajouter le modifs
            json.dump(dico, f, ensure_ascii=False, indent=2) # on y ajoute les ids en plus

    return ids # on renvoi la liste encodée des mots


In [6]:
def predire(list):
    """ prédire la note de la ligne de données"""
    tens = torch.tensor(list, dtype=torch.float32)
    tens = tens.to(device)
    tens = tens.unsqueeze(0)  # Ajouter une dimension pour le batch
    logits = model(tens)
    return logits

In [7]:
if __name__ == "__main__":
    print("demarage...")
    df = pd.read_csv("dataset.csv")
    lignes = len(df)
    print(lignes)
    print(df.shape)
    print(df.columns.tolist())
    epochs = 1
    print ("demarage de l'entrainement...")
    for epoch in range(epoch_depart, epochs):
        model.train()
        loss_totale = 0.0
        for l in range(lignes):
            ligne = selectionner(l)
            cible = ligne.pop(4)
            ligne = encode(ligne, "dico.json")
            pred = predire(ligne)
            # print("Prediction :", pred)
            # print("Cible :", cible)
            # print("Ligne encodée :", ligne)
            # print("NaN présents :", any(pd.isna(x) for x in ligne))
            cible = float(cible)  # Convertir la cible en float avant de créer le tenseur
            cible = torch.tensor([[float(cible)]], dtype=torch.float32, device=device)
            loss = criterion(pred, cible)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_totale += loss.item()

        loss_moyenne = loss_totale / lignes
        print(f"Époque {epoch + 1}/{epochs} — loss : {loss_moyenne:.4f}")
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict()
            }, "checkpoint.pt"
        )


demarage...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset.csv'